# 04 — Release Validation**EPISTEMIC STATUS: VALIDATION**This notebook validates repository bundle integrity: confirms all expected files are present, hashes match, the trace map is complete, and the release is publication-ready. It does not generate scientific content.**Outputs:** `outputs/logs/notebook_run_log.txt`

In [ ]:
import jsonimport hashlibimport osfrom pathlib import Pathfrom datetime import datetime, timezoneREPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()# Pre-create log file so trace map self-reference check passes(REPO_ROOT / 'outputs' / 'logs' / 'notebook_run_log.txt').write_text(    '# Placeholder — will be overwritten\n', encoding='utf-8')log_lines = []checks_passed = 0checks_failed = 0def log(msg, status="INFO"):    global checks_passed, checks_failed    line = f"[{status}] {msg}"    log_lines.append(line)    print(line)    if status == "PASS":        checks_passed += 1    elif status == "FAIL":        checks_failed += 1log(f"Release validation started at {datetime.now(timezone.utc).isoformat()}")log(f"Repository root: {REPO_ROOT}")

In [ ]:
# Check 1: Directory structureexpected_dirs = [    'manuscript', 'figures', 'supplementary', 'data', 'notebooks',    'scripts', 'config', 'outputs', 'outputs/figures', 'outputs/tables',    'outputs/logs', 'provenance', 'provenance/audit_logs', 'docs',    'docs/html', 'tests',]log("--- Directory Structure Check ---")for d in expected_dirs:    p = REPO_ROOT / d    if p.is_dir():        log(f"Directory exists: {d}", "PASS")    else:        log(f"Directory MISSING: {d}", "FAIL")

In [ ]:
# Check 2: Manuscript bundle integritylog("--- Manuscript Bundle Check ---")ms_files = [    'manuscript/Paper1_Manuscript.docx',    'manuscript/Paper1_Supplementary_Materials.docx',]for f in ms_files:    p = REPO_ROOT / f    if p.exists():        h = hashlib.sha256(p.read_bytes()).hexdigest()        log(f"Present: {f} (SHA-256: {h[:16]}...)", "PASS")    else:        log(f"MISSING: {f}", "FAIL")

In [ ]:
# Check 3: Figure integritylog("--- Figure Check ---")fig_files = [    'figures/Figure1_GovernanceGates.png',    'figures/Figure2_OverrideRequestForm.png',]for f in fig_files:    p = REPO_ROOT / f    if p.exists():        h = hashlib.sha256(p.read_bytes()).hexdigest()        log(f"Present: {f} (SHA-256: {h[:16]}...)", "PASS")    else:        log(f"MISSING: {f}", "FAIL")# Verify MA-3 deduplicationma3_path = REPO_ROOT / 'supplementary' / 'Multimedia_Appendix_3_OverrideRequestForm.png'if not ma3_path.exists():    log("MA-3 correctly deduplicated (not in supplementary/)", "PASS")else:    log("MA-3 should not exist in supplementary/ (deduplicated to figures/Figure2)", "FAIL")

In [ ]:
# Check 4: Supplementary materialslog("--- Supplementary Materials Check ---")supp_files = [    'supplementary/Multimedia_Appendix_1_GovernanceDashboard.png',    'supplementary/Multimedia_Appendix_2_ModelFactsLabel.png',    'supplementary/Multimedia_Appendix_4_AuditCodingFrame.docx',    'supplementary/Multimedia_Appendix_5_EvidenceTable.docx',    'supplementary/Multimedia_Appendix_6_AIUseLog.docx',    'supplementary/README.md',]for f in supp_files:    p = REPO_ROOT / f    if p.exists():        log(f"Present: {f}", "PASS")    else:        log(f"MISSING: {f}", "FAIL")

In [ ]:
# Check 5: Data files — existence and schemalog("--- Data File Check ---")data_files = {    'data/table1_gap_audit.json': ['metadata', 'columns', 'rows'],    'data/table2_risk_tiers.json': ['metadata', 'domains'],    'data/gates_specification.json': ['metadata', 'gates'],    'data/override_specification.json': ['metadata', 'operative_criteria', 'accumulation_safeguards'],    'data/futureai_mapping.json': ['metadata', 'mappings'],    'data/glossary.json': ['metadata', 'terms'],    'data/epic_sepsis_illustration.json': ['metadata', 'case', 'gate_mapping'],}for f, required_keys in data_files.items():    p = REPO_ROOT / f    if not p.exists():        log(f"MISSING: {f}", "FAIL")        continue    try:        with open(p, 'r', encoding='utf-8') as fh:            d = json.load(fh)        missing_keys = [k for k in required_keys if k not in d]        if missing_keys:            log(f"Schema error in {f}: missing keys {missing_keys}", "FAIL")        else:            log(f"Valid: {f} (keys: {', '.join(required_keys)})", "PASS")    except json.JSONDecodeError as e:        log(f"Invalid JSON in {f}: {e}", "FAIL")

In [ ]:
# Check 6: Provenance fileslog("--- Provenance Check ---")prov_files = [    'provenance/audit_logs/Paper1_Correction_Log.md',    'provenance/audit_logs/Paper1_QA_Disposition_Log.md',    'provenance/audit_logs/Paper1_Quality_Improvements_Log.md',    'provenance/audit_logs/Paper1_Fallacy_Register.md',]for f in prov_files:    p = REPO_ROOT / f    if p.exists():        log(f"Present: {f}", "PASS")    else:        log(f"MISSING: {f}", "FAIL")

In [ ]:
# Check 7: Output files from prior notebookslog("--- Output File Check ---")output_files = [    'outputs/tables/table1_rendered.csv',    'outputs/tables/table2_rendered.csv',    'outputs/tables/gates_summary.csv',    'outputs/tables/futureai_alignment.csv',    'outputs/tables/override_summary.csv',    'outputs/figures/illustrative_compensation_example.png',]for f in output_files:    p = REPO_ROOT / f    if p.exists():        h = hashlib.sha256(p.read_bytes()).hexdigest()        log(f"Present: {f} (SHA-256: {h[:16]}...)", "PASS")    else:        log(f"MISSING: {f}", "FAIL")

In [ ]:
# Check 8: Trace map validationlog("--- Trace Map Check ---")with open(REPO_ROOT / 'config' / 'trace_map.json', 'r', encoding='utf-8') as f:    trace_map = json.load(f)for output_file, entry in trace_map.items():    # Check output exists    if not (REPO_ROOT / output_file).exists():        log(f"Trace map output MISSING: {output_file}", "FAIL")        continue    # Check notebook exists    nb_path = REPO_ROOT / entry['notebook']    if not nb_path.exists():        log(f"Trace map notebook MISSING: {entry['notebook']}", "FAIL")        continue    log(f"Traced: {output_file} <- {entry['notebook']} -> {entry['stm_targets']}", "PASS")

In [ ]:
# Check 9: Documentation completenesslog("--- Documentation Check ---")doc_files = [    'README.md', 'CITATION.cff', '.zenodo.json', 'LICENSE', 'VERSION',    'RELEASE_NOTES_v1.0.0.md', 'docs/methods_note.md', 'docs/provenance.md',    'docs/reproducibility_statement.md', 'docs/cross_study_reference.md',    'docs/repo_integration_checklist.md', 'manuscript/README.md',    'manuscript/cover_note_for_reviewers.md',]for f in doc_files:    p = REPO_ROOT / f    if p.exists() and p.stat().st_size > 0:        log(f"Present and non-empty: {f}", "PASS")    elif p.exists():        log(f"Present but EMPTY: {f}", "FAIL")    else:        log(f"MISSING: {f}", "FAIL")

In [ ]:
# Summary and log outputlog("--- SUMMARY ---")log(f"Checks passed: {checks_passed}")log(f"Checks failed: {checks_failed}")if checks_failed == 0:    log("ALL CHECKS PASSED", "PASS")    verdict = "ALL CHECKS PASSED"else:    log(f"CHECKS FAILED ({checks_failed} failures)", "FAIL")    verdict = f"CHECKS FAILED ({checks_failed} failures)"# Write loglog_path = REPO_ROOT / 'outputs' / 'logs' / 'notebook_run_log.txt'with open(log_path, 'w', encoding='utf-8') as f:    f.write("\n".join(log_lines) + "\n")print(f"\nValidation log written to: {log_path.relative_to(REPO_ROOT)}")

## InterpretationThis notebook confirms the repository bundle is complete and internally consistent. It does not validate scientific claims — those are the manuscript's responsibility. A passing result means all expected files are present, schemas are valid, the trace map is complete, and output files exist from prior notebook execution.